# Reconocimiento de billetes peruanos (v1) — entrenamiento TFLite

Modelo MobileNetV3Small (transfer learning) para clasificar billetes de **S/10, S/20, S/50 y S/100** (series antigua y nueva, ambas caras).

**Dataset:** Caytuiro-Silva, N. E. et al. (2023). *Peruvian Banknotes*. Mendeley Data, V1. doi:10.17632/22jmt8xhpn.1 — licencia CC BY 4.0.

Estructura del ZIP: `training_set/` y `validation_set/` con 16 carpetas (`billete{10,20,50,100}_{anverso,reverso}_{antiguo,nuevo}`) y `testing_set/` con 261 fotos **sin etiquetas**. Por eso el test con métricas sale de la mitad de `validation_set` (la otra mitad es para validación).

Versión 1: aún sin S/200, monedas ni clase `fondo` (se agregan con fotos propias).

In [ ]:
# 1) Comprobar GPU y descargar el dataset (~934 MB)
import os, pathlib
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

URL = 'https://data.mendeley.com/public-files/datasets/22jmt8xhpn/files/ad4ba17a-8717-4223-ab4c-4b1b50452a59/file_downloaded'
if not os.path.exists('billetes.zip'):
    !wget -q -O billetes.zip "$URL"
!unzip -q -o billetes.zip -d billetes_raw

# Estructura: carpetas con imágenes y cuántas tiene cada una
conteo = {}
for p in pathlib.Path('billetes_raw').rglob('*'):
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and '__MACOSX' not in p.parts:
        conteo[p.parent.as_posix()] = conteo.get(p.parent.as_posix(), 0) + 1
for carpeta, n in sorted(conteo.items()):
    print(f'{n:5d}  {carpeta}')
print('Total imágenes:', sum(conteo.values()))

In [ ]:
# 2) Reorganizar en data/{train,val,test}/{b10,b20,b50,b100}
#    - Se juntan serie antigua/nueva y anverso/reverso en una sola clase por valor.
#    - training_set -> train. validation_set -> mitad val (early stopping) y mitad test (métricas).
#    - testing_set no trae etiquetas: solo se usa para ver ejemplos al final.
import re, shutil, hashlib, collections
RAW, DEST = pathlib.Path('billetes_raw'), pathlib.Path('data')

def valor_de(carpeta):  # 'billete100_anverso_antiguo' -> '100'
    m = re.search(r'(?<!\d)(100|50|20|10)(?!\d)', carpeta)
    return m.group(1) if m else None

def split_de(rel):
    r = '/' + rel.lower()
    if '/training_set/' in r:
        return 'train'
    if '/validation_set/' in r:  # reparto 50/50 determinista por nombre de archivo
        return 'val' if int(hashlib.md5(r.encode()).hexdigest(), 16) % 2 == 0 else 'test'
    return None

if DEST.exists(): shutil.rmtree(DEST)
resumen = collections.Counter()
for p in sorted(RAW.rglob('*')):
    if p.suffix.lower() not in ('.jpg', '.jpeg', '.png') or '__MACOSX' in p.parts:
        continue
    sp, v = split_de(p.relative_to(RAW).as_posix()), valor_de(p.parent.name)
    if sp is None or v is None:
        resumen[(p.parent.name, 'sin etiqueta', '-')] += 1
        continue
    resumen[(p.parent.name, sp, f'b{v}')] += 1
    d = DEST / sp / f'b{v}'
    d.mkdir(parents=True, exist_ok=True)
    os.link(p, d / f'{p.parent.name}_{p.name}')

for (carpeta, sp, clase), n in sorted(resumen.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    print(f'{carpeta:28s} -> {sp:12s} {clase:5s} ({n})')
for sp in ('train', 'val', 'test'):
    print(sp, {c.name: len(list(c.iterdir())) for c in sorted((DEST / sp).iterdir())})

In [ ]:
# 3) Datos + aumentos (rotación libre, zoom, brillo y contraste para simular iluminaciones)
IMG, BATCH = 224, 32
AUTOTUNE = tf.data.AUTOTUNE
kw = dict(image_size=(IMG, IMG), batch_size=BATCH, crop_to_aspect_ratio=True)  # recorte central, igual que la app
train_ds = tf.keras.utils.image_dataset_from_directory('data/train', shuffle=True, seed=42, **kw)
val_ds = tf.keras.utils.image_dataset_from_directory('data/val', shuffle=False, **kw)
class_names = train_ds.class_names
print('Clases:', class_names)

aug = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.5),
    tf.keras.layers.RandomZoom((-0.2, 0.2)),
    tf.keras.layers.RandomBrightness(0.4),
    tf.keras.layers.RandomContrast(0.4),
])
train_aug = train_ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_p = val_ds.prefetch(AUTOTUNE)

In [ ]:
# 4) Modelo: MobileNetV3Small preentrenado en ImageNet (recibe píxeles 0-255)
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet', pooling='avg')
base.trainable = False

inp = tf.keras.Input((IMG, IMG, 3))
x = base(inp, training=False)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inp, out)

# Pesos por clase: b50 tiene ~2 veces más fotos que b10
conteo = {c: len(os.listdir(f'data/train/{c}')) for c in class_names}
class_weight = {i: sum(conteo.values()) / (len(class_names) * conteo[c]) for i, c in enumerate(class_names)}
print('Fotos de train:', conteo, '| pesos:', {c: round(class_weight[i], 2) for i, c in enumerate(class_names)})

parar = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = model.fit(train_aug, validation_data=val_p, epochs=12, callbacks=[parar], class_weight=class_weight)

In [ ]:
# 5) Fine-tuning de toda la red con learning rate bajo
base.trainable = True
model.compile(tf.keras.optimizers.Adam(1e-5), 'sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_aug, validation_data=val_p, epochs=10, callbacks=[parar], class_weight=class_weight)

import matplotlib.pyplot as plt
acc = h1.history['accuracy'] + h2.history['accuracy']
vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
plt.plot(acc, label='train'); plt.plot(vacc, label='val')
plt.axvline(len(h1.history['accuracy']) - 0.5, ls='--', c='gray', label='inicio fine-tuning')
plt.xlabel('época'); plt.ylabel('accuracy'); plt.legend(); plt.title('Curvas de entrenamiento')
plt.savefig('curvas.png', bbox_inches='tight', dpi=150); plt.show()

In [ ]:
# 6) Exportar a TFLite (cuantización dinámica) + etiquetas
model.export('modelo_sm')
conv = tf.lite.TFLiteConverter.from_saved_model('modelo_sm')
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = conv.convert()
open('soles.tflite', 'wb').write(tflite)
open('labels.txt', 'w').write('\n'.join(class_names))
print(f'soles.tflite: {len(tflite) / 1e6:.2f} MB | etiquetas: {class_names}')

In [ ]:
# 7) Evaluar el .tflite final (lo mismo que correrá en el celular): accuracy + matriz de confusión
import numpy as np
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter

interp = Interpreter(model_path='soles.tflite')
interp.allocate_tensors()
i_in, i_out = interp.get_input_details()[0], interp.get_output_details()[0]

def predecir(ruta):
    img = ImageOps.fit(Image.open(ruta).convert('RGB'), (IMG, IMG))  # recorte central
    interp.set_tensor(i_in['index'], np.asarray(img, np.float32)[None])
    interp.invoke()
    return interp.get_tensor(i_out['index'])[0]

def evaluar(carpeta, titulo, umbral=0.85):
    y_true, y_pred, conf = [], [], []
    for i, c in enumerate(class_names):
        for f in sorted(pathlib.Path(carpeta, c).glob('*')):
            p = predecir(f)
            y_true.append(i); y_pred.append(int(p.argmax())); conf.append(float(p.max()))
    y_true, y_pred, conf = map(np.array, (y_true, y_pred, conf))
    print(f'=== {titulo}: {len(y_true)} imágenes | accuracy = {accuracy_score(y_true, y_pred):.4f}')
    print(classification_report(y_true, y_pred, labels=range(len(class_names)),
                                target_names=class_names, digits=4, zero_division=0))
    habla = conf >= umbral  # como en la app: solo habla si la confianza supera el umbral
    if habla.any():
        print(f'Umbral {umbral}: habla en {habla.mean():.1%} de los casos; '
              f'se equivoca en {(y_pred[habla] != y_true[habla]).mean():.2%} de lo que dice')
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues')
    plt.title(f'Matriz de confusión — {titulo}')
    plt.savefig(f'cm_{titulo}.png', bbox_inches='tight', dpi=150); plt.show()

evaluar('data/test', 'test')
evaluar('data/val', 'val')

In [ ]:
# 8) Ejemplos visuales sobre testing_set (no tiene etiquetas: solo para ver qué predice)
import random
sin_etiqueta = sorted(p for p in RAW.glob('**/testing_set/*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
plt.figure(figsize=(12, 9))
for k, f in enumerate(random.Random(0).sample(sin_etiqueta, min(12, len(sin_etiqueta)))):
    p = predecir(f)
    plt.subplot(3, 4, k + 1); plt.imshow(Image.open(f)); plt.axis('off')
    plt.title(f'{f.name}: {class_names[p.argmax()]} ({p.max():.0%})')
plt.tight_layout(); plt.savefig('ejemplos_testing_set.png', dpi=120); plt.show()

In [ ]:
# 9) Más adelante, con tus fotos separadas por iluminación: test_propio/<iluminacion>/<clase>/*.jpg
# for cond in sorted(os.listdir('test_propio')):
#     evaluar(f'test_propio/{cond}', cond)

In [ ]:
# 10) Descargar el modelo y los resultados a tu PC
# from google.colab import files
# for f in ['soles.tflite', 'labels.txt', 'curvas.png', 'cm_test.png', 'cm_val.png', 'ejemplos_testing_set.png']:
#     files.download(f)